# CNV GWAS of height and BMI - All of Us Researcher Workbench
Same code base as the UKBB run (branch `cnv-gwas` of jsebat/cnv-growth-bmi); only `configs/aou_workbench.json` differs. Controlled Tier, Python kernel. Read `HANDOFF_AoU_Milad` first.

Individual-level files stay on this disk / the workspace bucket. Only `export_aggregate/` (and meta tables) may be downloaded, after the small-cell rule applied by the export step. Clear large cell outputs before saving the notebook (egress alerts).

In [ ]:
import os, subprocess, json
print(os.environ.get('WORKSPACE_CDR'), os.environ.get('WORKSPACE_BUCKET'), os.environ.get('GOOGLE_PROJECT'))
ROOT = os.path.expanduser('~/cnv-growth-bmi/cnv_gwas')
if not os.path.exists(ROOT):
    subprocess.run(['git', 'clone', '-b', 'cnv-gwas', 'https://github.com/jsebat/cnv-growth-bmi.git', os.path.expanduser('~/cnv-growth-bmi')], check=True)
CODE = f'{ROOT}/cnv_anthro'
for d in ['data', 'results']:
    os.makedirs(os.path.expanduser(f'~/cnv_anthro_run/{d}'), exist_ok=True)

## 0. Resources bundle
Upload `cnv_gwas_resources_2026-11-15.tar.gz` (from Jon / Expanse `ukbb_anthro_cnv/aou_bundle/`) to the workspace bucket first.

In [ ]:
BUNDLE = 'cnv_gwas_resources_2026-11-15.tar.gz'
if not os.path.exists(f'{ROOT}/resources'):
    subprocess.run(f"gsutil cp $WORKSPACE_BUCKET/{BUNDLE} {ROOT}/ && cd {ROOT} && tar xzf {BUNDLE}", shell=True, check=True)
print(os.listdir(f'{ROOT}/resources'))

## 1. Fill the two data paths
From the Workbench page on how the genomic data are organised (version matching this workspace's CDR): the srWGS **GATK-SV joint-called VCF** and **ancestry_preds.tsv**. The ancestry file is copied locally; the VCF is streamed with `gsutil -u $GOOGLE_PROJECT cat`.

In [ ]:
cfg_path = f'{CODE}/configs/aou_workbench.json'
cfg = json.load(open(cfg_path))
SV_VCF = 'gs://...'          # <- fill
ANCESTRY_GS = 'gs://...'     # <- fill
anc = os.path.expanduser('~/cnv_anthro_run/data/ancestry_preds.tsv')
subprocess.run(['gsutil', '-u', os.environ['GOOGLE_PROJECT'], 'cp', ANCESTRY_GS, anc], check=True)
cfg['prep_commands'][0][2] = SV_VCF
cfg['prep_commands'][1][cfg['prep_commands'][1].index('--ancestry') + 1] = anc
json.dump(cfg, open(cfg_path, 'w'), indent=1)

## 2. Prep (SV VCF -> CNV table; OMOP -> phenotype table)
The VCF pass is the slow part; run on a large VM or with `workbench/dsub_run.sh prep`.

In [ ]:
!cd {CODE} && python run_pipeline.py --config configs/aou_workbench.json --steps prep
print(open(os.path.expanduser('~/cnv_anthro_run/results/aou_pheno_summary.json')).read())

## 3. Probe GWAS, S9 alleles, growth-locus effects - every stratum (all, eur, afr, amr, eas, sas, mid, female, male)

In [ ]:
!cd {CODE} && python run_pipeline.py --config configs/aou_workbench.json --steps gwas,plots,alleles,loci

## 4. Genome-wide grid, call-size split, joint models (shared UKBB shifts; strata all, eur, afr, amr, female, male)
Several hours on 64-96 vCPUs. Run `all` first. For long runs use a terminal (`nohup ... &`) or dsub.

In [ ]:
!cd {CODE} && python run_pipeline.py --config configs/aou_workbench.json --steps grid --strata all --jobs 24

In [ ]:
!cd {CODE} && python run_pipeline.py --config configs/aou_workbench.json --steps grid --strata eur,afr,amr,female,male --jobs 24

## 5. Export aggregate results (small-cell rule + cross-stratum suppression)
Check `EXPORT_MANIFEST.txt` (size, withheld rows) before downloading. Ask AoU support for the current egress size limit.

In [ ]:
!cd {CODE} && python run_pipeline.py --config configs/aou_workbench.json --steps export
print(open(os.path.expanduser('~/cnv_anthro_run/export_aggregate/EXPORT_MANIFEST.txt')).read()[:600])

## 6. (Recommended) meta-analysis with UKBB inside the Workbench
Upload the UKBB aggregate export (from Jon) to the bucket and unpack it to `~/ukbb_export`. Only the small meta tables then need to leave.

In [ ]:
UK = os.path.expanduser('~/ukbb_export'); EX = os.path.expanduser('~/cnv_anthro_run/export_aggregate'); MO = os.path.expanduser('~/cnv_anthro_run/meta')
for t in ['height', 'bmi']:
    for s in ['all', 'female', 'male']:
        subprocess.run(['python', 'meta_sumstats.py', '--cohort', f'UKBB={UK}/{s}', '--cohort', f'AoU={EX}/{s}', '--trait', t, '--out', f'{MO}/{t}_{s}', '--ref_label', 'UKBB'], cwd=CODE, check=True)
        for d in ['grid_univariate', 'grid_callsize', 'joint_2026-11-15']:
            if os.path.exists(f'{EX}/{s}/grid/{d}/{t}') and os.path.exists(f'{UK}/{s}/grid/{d}/{t}'):
                subprocess.run(['python', 'meta_grid.py', '--cohort', f'UKBB={UK}/{s}/grid/{d}/{t}', '--cohort', f'AoU={EX}/{s}/grid/{d}/{t}', '--out', f'{MO}/meta_{d}_{t}_{s}.tsv'], cwd=CODE, check=True)

Ancestry strata are named differently in the two cohorts (UKBB: EUR, AFR, ASN, ...; AoU: eur, afr, eas/sas, ...): meta them explicitly (e.g. UKBB EUR with AoU eur).

In [ ]:
!cd ~/cnv_anthro_run && tar czf aou_cnv_export_$(date +%Y%m%d).tar.gz --exclude='*.npz' export_aggregate meta && ls -la *.tar.gz